# eICU AKI Prediction with Trajectory Modeling
Analysis of creatinine trajectory features for predicting AKI Stage 3

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✓ Imports successful')

✓ Imports successful


## Load Data

In [2]:
dataset = pd.read_csv('../../../results/eicu/aki/aki_trajectory_probs.csv')
print(f"✓ Loaded merged dataset: {len(dataset):,} samples")
print(f"  Unique patients: {dataset['stay_id'].nunique():,}")
print(f"  Outcome rate: {dataset['target_aki_stage3'].mean():.1%}")
print(f"  Columns: {len(dataset.columns)}")

✓ Loaded merged dataset: 523,156 samples
  Unique patients: 70,081
  Outcome rate: 1.0%
  Columns: 58


In [3]:
creatinine_ts = pd.read_csv('../../../results/eicu/aki/creatinine_timeseries.csv')
print(f"✓ Loaded creatinine time series: {len(creatinine_ts):,} measurements")
print(f"  Patients: {creatinine_ts['stay_id'].nunique():,}")

✓ Loaded creatinine time series: 717,361 measurements
  Patients: 57,056


## Feature Engineering

In [4]:
LOOKBACK_DAYS = 7
summary_list = []

for patient_id, group_df in creatinine_ts.groupby('stay_id'):
    group_df = group_df.sort_values('time_days')
    for current_day in group_df['time_day'].unique():
        window_start = current_day - LOOKBACK_DAYS
        window_data = group_df[group_df['time_day'].between(window_start, current_day, inclusive='both')]
        if len(window_data) > 0:
            summary_list.append({
                'stay_id': patient_id,
                'time_day': current_day,
                'creat_mean_7d': window_data['creatinine'].mean(),
                'creat_max_7d': window_data['creatinine'].max(),
                'creat_min_7d': window_data['creatinine'].min(),
                'creat_change_7d': window_data['creatinine'].iloc[-1] - window_data['creatinine'].iloc[0],
                'creat_linear_trend_7d': np.polyfit(window_data['time_days'], window_data['creatinine'], 1)[0] if len(window_data) > 1 else 0,
                'creat_std_7d': window_data['creatinine'].std() if len(window_data) > 1 else 0
            })

creat_summary = pd.DataFrame(summary_list)
print(f'✓ Created 7-day creatinine summary: {len(creat_summary):,} timepoints')

✓ Created 7-day creatinine summary: 529,871 timepoints


## Merge Features

In [5]:
dataset['prob_worsening'] = dataset['prob_gradual_increase'] + dataset['prob_rapid_increase']

dataset = dataset.merge(
    creat_summary,
    on=['stay_id', 'time_day'],
    how='left'
)

traj_cols = ['prob_stable', 'prob_gradual_increase', 'prob_rapid_increase', 'prob_worsening']
dataset[traj_cols] = dataset.groupby('stay_id')[traj_cols].ffill(limit=2)

print(f"✓ Merged summary statistics: {len(dataset):,} samples, {len(dataset.columns)} columns")
print(f"Missing trajectory probabilities: {dataset['prob_stable'].isna().sum():,} ({100*dataset['prob_stable'].isna().mean():.1f}%)")

✓ Merged summary statistics: 523,156 samples, 65 columns
Missing trajectory probabilities: 224,906 (43.0%)


## Define Feature Sets

In [6]:
summary_cols = ['creat_mean_7d', 'creat_max_7d', 'creat_min_7d', 'creat_change_7d', 'creat_linear_trend_7d', 'creat_std_7d']
static_cols = [c for c in ['baseline_creatinine', 'creat_fold_change', 'creat_above_normal', 'age', 'gender'] if c in dataset.columns]

exclude_cols = {'stay_id', 'time_day', 'target_aki_stage3', 'dominant_traj'}
exclude_cols.update(traj_cols)
exclude_cols.update(summary_cols)
exclude_cols.update(static_cols)
dynamic_cols = [c for c in dataset.columns if c not in exclude_cols and dataset[c].dtype != 'object']

feature_sets = {
    'Trajectory Only': traj_cols,
    'Summary Stats Only': summary_cols,
    'Trajectory + Summary Stats': traj_cols + summary_cols,
    'Static Only': static_cols,
    'Trajectory + Static': traj_cols + static_cols,
    'Summary Stats + Static': summary_cols + static_cols,
    'Trajectory + Summary Stats + Static': traj_cols + summary_cols + static_cols,
}

if len(dynamic_cols) > 0:
    feature_sets['Static + Dynamic'] = static_cols + dynamic_cols
    feature_sets['Trajectory + Static + Dynamic'] = traj_cols + static_cols + dynamic_cols
    feature_sets['Summary Stats + Static + Dynamic'] = summary_cols + static_cols + dynamic_cols
    feature_sets['Trajectory + Summary Stats + Static + Dynamic'] = traj_cols + summary_cols + static_cols + dynamic_cols

for name, features in feature_sets.items():
    print(f'{name:40s}: {len(features):2d} features')

Trajectory Only                         :  4 features
Summary Stats Only                      :  6 features
Trajectory + Summary Stats              : 10 features
Static Only                             :  4 features
Trajectory + Static                     :  8 features
Summary Stats + Static                  : 10 features
Trajectory + Summary Stats + Static     : 14 features
Static + Dynamic                        : 51 features
Trajectory + Static + Dynamic           : 55 features
Summary Stats + Static + Dynamic        : 57 features
Trajectory + Summary Stats + Static + Dynamic: 61 features


## Train Models

In [7]:
dataset_clean = dataset.dropna(subset=['target_aki_stage3']).copy()
dataset_clean['gender'] = dataset_clean['gender'].map({'M': 1, 'F': 0})

y = dataset_clean['target_aki_stage3']
groups = dataset_clean['stay_id']

print(f'Dataset: {len(y):,} samples, {groups.nunique():,} patients')
print(f'Positive rate: {y.mean():.1%}')

Dataset: 523,156 samples, 70,081 patients
Positive rate: 1.0%


In [ ]:
models_to_evaluate = {
    'XGBoost': lambda pos_weight, seed: XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        scale_pos_weight=pos_weight,
        random_state=seed,
        eval_metric='logloss',
    ),
    'Logistic Regression': lambda pos_weight, seed: LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=seed,
        solver='lbfgs'
    ),
    'Random Forest': lambda pos_weight, seed: RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight='balanced',
        random_state=seed,
        n_jobs=4
    ),
    'Gradient Boosting': lambda pos_weight, seed: HistGradientBoostingClassifier(
        max_bins=225,
        max_depth=3,
        learning_rate=0.1,
        class_weight='balanced',
        random_state=seed
    )
}

n_repeats = 10
n_folds = 5
results = {model_name: {} for model_name in models_to_evaluate.keys()}

print(f"\nTraining {n_repeats}-Repeat {n_folds}-Fold CV with {len(models_to_evaluate)} models:\n")

for model_name, model_fn in models_to_evaluate.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)

    dataset_model = dataset_clean.copy()
    if model_name not in ['XGBoost', 'Gradient Boosting']:
        dataset_model[traj_cols] = dataset_model[traj_cols].fillna(0)
        dataset_model[summary_cols] = dataset_model[summary_cols].fillna(0)

    for feature_set_name, feature_cols in feature_sets.items():
        print(f"{feature_set_name}...")
        fold_metrics = {'roc_auc': [], 'avg_precision': [], 'y_true': [], 'y_pred': []}

        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920+repeat).permutation(len(dataset_model))
            dataset_repeat = dataset_model.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)

            gkf = GroupKFold(n_splits=n_folds)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                if model_name not in ['XGBoost', 'Gradient Boosting']:
                    imputer = SimpleImputer(strategy='median')
                    X_train_imputed = imputer.fit_transform(X_train)
                    X_test_imputed = imputer.transform(X_test)
                else:
                    X_train_imputed = X_train
                    X_test_imputed = X_test

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
                model = model_fn(scale_pos_weight, 920+repeat)
                model.fit(X_train_scaled, y_train)

                y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

                fold_metrics['roc_auc'].append(roc_auc_score(y_test, y_pred_proba))
                fold_metrics['avg_precision'].append(average_precision_score(y_test, y_pred_proba))
                fold_metrics['y_true'].extend(y_test)
                fold_metrics['y_pred'].extend(y_pred_proba)

        results[model_name][feature_set_name] = fold_metrics
        print(f"  AUROC: {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}")
        print(f"  AUPR:  {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}\n")


Training 10-Repeat 5-Fold CV with 4 models:


Model: XGBoost
Trajectory Only...
  AUROC: 0.723 ± 0.004
  AUPR:  0.041 ± 0.005

Summary Stats Only...
  AUROC: 0.876 ± 0.003
  AUPR:  0.245 ± 0.009

Trajectory + Summary Stats...
  AUROC: 0.880 ± 0.002
  AUPR:  0.264 ± 0.010

Static Only...
  AUROC: 0.875 ± 0.002
  AUPR:  0.131 ± 0.006

Trajectory + Static...
  AUROC: 0.890 ± 0.002
  AUPR:  0.187 ± 0.010

Summary Stats + Static...
  AUROC: 0.901 ± 0.004
  AUPR:  0.252 ± 0.010

Trajectory + Summary Stats + Static...
  AUROC: 0.902 ± 0.003
  AUPR:  0.276 ± 0.010

Static + Dynamic...
  AUROC: 0.911 ± 0.005
  AUPR:  0.230 ± 0.012

Trajectory + Static + Dynamic...
  AUROC: 0.915 ± 0.004
  AUPR:  0.262 ± 0.015

Summary Stats + Static + Dynamic...
  AUROC: 0.920 ± 0.004
  AUPR:  0.321 ± 0.013

Trajectory + Summary Stats + Static + Dynamic...
  AUROC: 0.921 ± 0.004
  AUPR:  0.336 ± 0.014


Model: Logistic Regression
Trajectory Only...
  AUROC: 0.716 ± 0.005
  AUPR:  0.026 ± 0.003

Summary Stats O

## Visualize Results

In [ ]:
for model_name in models_to_evaluate.keys():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'{model_name} - All Feature Sets', fontsize=14, fontweight='bold')
    colors = plt.cm.tab20(np.linspace(0, 1, len(results[model_name])))

    for idx, (name, metrics) in enumerate(results[model_name].items()):
        fpr, tpr, _ = roc_curve(metrics['y_true'], metrics['y_pred'])
        auc = np.mean(metrics['roc_auc'])
        ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=colors[idx], linewidth=2)

        precision, recall, _ = precision_recall_curve(metrics['y_true'], metrics['y_pred'])
        ap = np.mean(metrics['avg_precision'])
        ax2.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=colors[idx], linewidth=2)

    ax1.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.legend(loc='lower right', fontsize=8)
    ax1.grid(True, alpha=0.3)

    baseline = y.mean()
    ax2.axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.3f})', linewidth=1)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.legend(loc='upper right', fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
print('Model Comparison Across All Feature Sets:')
summary_rows = []
for model_name, model_results in results.items():
    for feature_set, metrics in model_results.items():
        summary_rows.append({
            'Model': model_name,
            'Feature Set': feature_set,
            'AUROC Mean': np.mean(metrics['roc_auc']),
            'AUROC Std': np.std(metrics['roc_auc']),
            'AUPR Mean': np.mean(metrics['avg_precision']),
            'AUPR Std': np.std(metrics['avg_precision'])
        })
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(['AUROC Mean', 'AUPR Mean'], ascending=False)
summary_df.head(10)